# Contextual Data Fusion and Feature Engineering

## Objective

This notebook prepares machine telemetry signals for contextual feature engineering.

Tasks:

- Load telemetry dataset
- Identify sensor signals
- Create feature engineering workflow
- Prepare for rolling statistics generation
- Build the foundation for contextual data fusion

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/raw/ai4i2020.csv")

print(df.shape)

df.head()

(10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
sensor_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

sensor_cols

['Air temperature [K]',
 'Process temperature [K]',
 'Rotational speed [rpm]',
 'Torque [Nm]',
 'Tool wear [min]']

In [4]:
sensor_summary = df[sensor_cols].describe().T

sensor_summary

,count,mean,std,min,25%,50%,75%,max
Air temperature [K],10000.0,300.00493,2.000259,295.3,298.3,300.1,301.5,304.5
Process temperature [K],10000.0,310.00556,1.483734,305.7,308.8,310.1,311.1,313.8
Rotational speed [rpm],10000.0,1538.77610,179.284096,1168.0,1423.0,1503.0,1612.0,2886.0
Torque [Nm],10000.0,39.98691,9.968934,3.8,33.2,40.1,46.8,76.6
Tool wear [min],10000.0,107.95100,63.654147,0.0,53.0,108.0,162.0,253.0


In [5]:
variability_df = pd.DataFrame(
    {
        "Mean": df[sensor_cols].mean(),
        "Std": df[sensor_cols].std(),
        "Variance": df[sensor_cols].var(),
    }
)

variability_df

,Mean,Std,Variance
Air temperature [K],300.00493,2.000259,4.001035
Process temperature [K],310.00556,1.483734,2.201467
Rotational speed [rpm],1538.77610,179.284096,32142.787047
Torque [Nm],39.98691,9.968934,99.379640
Tool wear [min],107.95100,63.654147,4051.850384


In [6]:
telemetry_corr = df[sensor_cols].corr()

telemetry_corr

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
Air temperature [K],1.000000,0.876107,0.022670,-0.013778,0.013853
Process temperature [K],0.876107,1.000000,0.019277,-0.014061,0.013488
Rotational speed [rpm],0.022670,0.019277,1.000000,-0.875027,0.000223
Torque [Nm],-0.013778,-0.014061,-0.875027,1.000000,-0.003093
Tool wear [min],0.013853,0.013488,0.000223,-0.003093,1.000000


In [7]:
telemetry_df = df.copy()

print(telemetry_df.shape)

(10000, 14)


In [8]:
telemetry_df.to_csv("../data/processed/telemetry_base.csv", index=False)

print("Telemetry dataset saved.")

Telemetry dataset saved.


### Telemetry Preparation Findings

- Five primary sensor signals were identified.
- Sensor variability differs significantly across measurements.
- Rotational speed exhibits the largest variance.
- These signals will be transformed into rolling, lag, and trend features in subsequent stages.

## Rolling Window Feature Engineering

Rolling statistics capture short-term operational trends and variability in machine telemetry.

Features generated:

- Rolling Mean
- Rolling Standard Deviation
- Rolling Variance

Window Size: 10 observations

In [9]:
telemetry_df = pd.read_csv("../data/processed/telemetry_base.csv")

print(telemetry_df.shape)

(10000, 14)


In [10]:
for col in sensor_cols:

    telemetry_df[f"{col}_roll_mean"] = telemetry_df[col].rolling(window=10).mean()

In [11]:
for col in sensor_cols:

    telemetry_df[f"{col}_roll_std"] = telemetry_df[col].rolling(window=10).std()

In [12]:
for col in sensor_cols:

    telemetry_df[f"{col}_roll_var"] = telemetry_df[col].rolling(window=10).var()

In [13]:
print(telemetry_df.shape)

(10000, 29)


In [14]:
telemetry_df.isnull().sum().sort_values(ascending=False).head(20)

Torque [Nm]_roll_var                 9
Air temperature [K]_roll_var         9
Tool wear [min]_roll_std             9
Torque [Nm]_roll_std                 9
Process temperature [K]_roll_var     9
Rotational speed [rpm]_roll_std      9
Process temperature [K]_roll_std     9
Tool wear [min]_roll_mean            9
Air temperature [K]_roll_std         9
Torque [Nm]_roll_mean                9
Rotational speed [rpm]_roll_mean     9
Process temperature [K]_roll_mean    9
Air temperature [K]_roll_mean        9
Tool wear [min]_roll_var             9
Rotational speed [rpm]_roll_var      9
OSF                                  0
TWF                                  0
Machine failure                      0
PWF                                  0
HDF                                  0
dtype: int64

In [15]:
telemetry_df = telemetry_df.dropna()

print(telemetry_df.shape)

(9991, 29)


In [16]:
rolling_cols = [col for col in telemetry_df.columns if "roll" in col]

display(rolling_cols[:10])

print(len(rolling_cols))

['Air temperature [K]_roll_mean',
 'Process temperature [K]_roll_mean',
 'Rotational speed [rpm]_roll_mean',
 'Torque [Nm]_roll_mean',
 'Tool wear [min]_roll_mean',
 'Air temperature [K]_roll_std',
 'Process temperature [K]_roll_std',
 'Rotational speed [rpm]_roll_std',
 'Torque [Nm]_roll_std',
 'Tool wear [min]_roll_std']

15


In [17]:
telemetry_df.to_csv("../data/processed/telemetry_rolling_features.csv", index=False)

print("Rolling feature dataset saved.")

Rolling feature dataset saved.


### Rolling Feature Findings

Rolling statistics provide localized summaries of machine behavior.

These features capture:

- Short-term trends
- Operational variability
- Signal stability

and are expected to improve predictive maintenance performance compared to raw telemetry alone.

## Lag Feature Engineering

Lag features capture historical machine states by incorporating previous sensor readings into the current observation.

Generated Features:

- Lag 1
- Lag 2

for all primary telemetry signals.

In [18]:
telemetry_df = pd.read_csv("../data/processed/telemetry_rolling_features.csv")

print(telemetry_df.shape)

(9991, 29)


In [19]:
sensor_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

In [20]:
for col in sensor_cols:

    telemetry_df[f"{col}_lag1"] = telemetry_df[col].shift(1)

In [21]:
for col in sensor_cols:

    telemetry_df[f"{col}_lag2"] = telemetry_df[col].shift(2)

In [22]:
print(telemetry_df.shape)

(9991, 39)


In [23]:
telemetry_df.isnull().sum().sort_values(ascending=False).head(10)

Rotational speed [rpm]_lag2     2
Torque [Nm]_lag2                2
Process temperature [K]_lag2    2
Air temperature [K]_lag2        2
Tool wear [min]_lag2            2
Process temperature [K]_lag1    1
Air temperature [K]_lag1        1
Torque [Nm]_lag1                1
Rotational speed [rpm]_lag1     1
Tool wear [min]_lag1            1
dtype: int64

In [24]:
telemetry_df = telemetry_df.dropna()

print(telemetry_df.shape)

(9989, 39)


In [25]:
lag_cols = [col for col in telemetry_df.columns if "lag" in col]

print(len(lag_cols))

lag_cols

10


['Air temperature [K]_lag1',
 'Process temperature [K]_lag1',
 'Rotational speed [rpm]_lag1',
 'Torque [Nm]_lag1',
 'Tool wear [min]_lag1',
 'Air temperature [K]_lag2',
 'Process temperature [K]_lag2',
 'Rotational speed [rpm]_lag2',
 'Torque [Nm]_lag2',
 'Tool wear [min]_lag2']

In [26]:
telemetry_df.to_csv("../data/processed/telemetry_lag_features.csv", index=False)

print("Lag feature dataset saved.")

Lag feature dataset saved.


### Lag Feature Findings

Lag features preserve historical machine behavior and provide temporal context.

These features allow the model to learn relationships between previous operating conditions and future machine failures.

## Change and Trend Features

Change features measure how sensor values evolve over time.

These features help identify abnormal operational behavior and rapid shifts that may precede machine failure.

In [27]:
telemetry_df = pd.read_csv("../data/processed/telemetry_lag_features.csv")

print(telemetry_df.shape)

(9989, 39)


In [28]:
for col in sensor_cols:

    telemetry_df[f"{col}_change"] = telemetry_df[col].diff()

In [29]:
print(telemetry_df.shape)

(9989, 44)


In [30]:
telemetry_df.isnull().sum().sort_values(ascending=False).head(10)

Air temperature [K]_change        1
Rotational speed [rpm]_change     1
Torque [Nm]_change                1
Tool wear [min]_change            1
Process temperature [K]_change    1
Process temperature [K]           0
Torque [Nm]                       0
Rotational speed [rpm]            0
UDI                               0
Product ID                        0
dtype: int64

In [31]:
telemetry_df = telemetry_df.dropna()

print(telemetry_df.shape)

(9988, 44)


In [32]:
change_cols = [col for col in telemetry_df.columns if "_change" in col]

print(len(change_cols))

change_cols

5


['Air temperature [K]_change',
 'Process temperature [K]_change',
 'Rotational speed [rpm]_change',
 'Torque [Nm]_change',
 'Tool wear [min]_change']

In [33]:
telemetry_df[change_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Air temperature [K]_change,9988.0,0.000040,0.068221,-0.3,0.0,0.0,0.0,0.2
Process temperature [K]_change,9988.0,-0.000040,0.079814,-0.3,-0.1,0.0,0.1,0.3
Rotational speed [rpm]_change,9988.0,0.007709,252.682332,-1478.0,-133.0,-1.0,134.0,1449.0
Torque [Nm]_change,9988.0,-0.000410,14.066026,-50.5,-9.3,0.0,9.3,52.2
Tool wear [min]_change,9988.0,0.000100,23.727376,-253.0,2.0,2.0,3.0,5.0


In [34]:
telemetry_df.to_csv("../data/processed/telemetry_engineered.csv", index=False)

print("Engineered telemetry dataset saved.")

Engineered telemetry dataset saved.


### Trend Feature Findings

Change features capture the rate and direction of sensor evolution.

These variables provide temporal context that is not present in raw telemetry readings alone and are expected to enhance predictive maintenance performance.

## Engineered Dataset Validation

The engineered telemetry dataset is validated before contextual feature integration.

Validation checks:

- Dataset dimensions
- Missing values
- Feature inventory
- Numerical consistency

In [35]:
telemetry_df = pd.read_csv("../data/processed/telemetry_engineered.csv")

print(telemetry_df.shape)

(9988, 44)


In [36]:
missing_values = telemetry_df.isnull().sum().sum()

print("Total Missing Values:", missing_values)

Total Missing Values: 0


In [37]:
raw_features = [
    col
    for col in telemetry_df.columns
    if ("roll" not in col and "lag" not in col and "_change" not in col)
]

rolling_features = [col for col in telemetry_df.columns if "roll" in col]

lag_features = [col for col in telemetry_df.columns if "lag" in col]

change_features = [col for col in telemetry_df.columns if "_change" in col]

In [38]:
feature_summary = pd.DataFrame(
    {
        "Feature Type": ["Raw", "Rolling", "Lag", "Change"],
        "Count": [
            len(raw_features),
            len(rolling_features),
            len(lag_features),
            len(change_features),
        ],
    }
)

feature_summary

,Feature Type,Count
0,Raw,14
1,Rolling,15
2,Lag,10
3,Change,5


In [39]:
telemetry_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9988 entries, 0 to 9987
Data columns (total 44 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   UDI                                9988 non-null   int64  
 1   Product ID                         9988 non-null   str    
 2   Type                               9988 non-null   str    
 3   Air temperature [K]                9988 non-null   float64
 4   Process temperature [K]            9988 non-null   float64
 5   Rotational speed [rpm]             9988 non-null   int64  
 6   Torque [Nm]                        9988 non-null   float64
 7   Tool wear [min]                    9988 non-null   int64  
 8   Machine failure                    9988 non-null   int64  
 9   TWF                                9988 non-null   int64  
 10  HDF                                9988 non-null   int64  
 11  PWF                                9988 non-null   int64  
 12  OSF

In [40]:
print(f"Rows: {telemetry_df.shape[0]}")

print(f"Columns: {telemetry_df.shape[1]}")

Rows: 9988
Columns: 44


In [41]:
telemetry_df.to_csv("../data/processed/telemetry_features.csv", index=False)

print("Validated telemetry dataset saved.")

Validated telemetry dataset saved.


### Validation Findings

- No missing values remain.
- All engineered features were generated successfully.
- The dataset now contains raw, rolling, lag, and change-based telemetry features.
- The validated dataset is ready for contextual data fusion.

In [42]:
df = pd.read_csv("../data/processed/telemetry_features.csv")

print(df.shape)

(9988, 44)


In [43]:
np.random.seed(42)

In [44]:
df["ambient_humidity"] = np.random.normal(loc=65, scale=10, size=len(df))

In [45]:
df["energy_load_index"] = np.random.uniform(40, 100, len(df))

In [46]:
df["production_demand"] = np.random.uniform(50, 150, len(df))

In [47]:
df["days_since_maintenance"] = np.random.randint(0, 180, len(df))

In [48]:
df["shift"] = np.random.choice([0, 1], len(df))

In [49]:
context_cols = [
    "ambient_humidity",
    "energy_load_index",
    "production_demand",
    "days_since_maintenance",
    "shift",
]

df[context_cols].describe()

,ambient_humidity,energy_load_index,production_demand,days_since_maintenance,shift
count,9988.000000,9988.000000,9988.000000,9988.000000,9988.000000
mean,64.978646,70.431452,99.672996,89.228474,0.496396
std,10.033366,17.363538,28.718016,52.190789,0.500012
min,25.775997,40.002887,50.006750,0.000000,0.000000
25%,58.281110,55.471277,74.745858,44.000000,0.000000
50%,64.969197,70.680500,99.666650,90.000000,0.000000
75%,71.710809,85.438879,124.213912,134.000000,1.000000
max,104.262377,99.995490,149.973208,179.000000,1.000000


In [50]:
print(df.shape)

(9988, 49)


In [51]:
df.to_csv("../data/processed/context_base.csv", index=False)

print("Contextual dataset saved.")

Contextual dataset saved.


### Contextual Feature Creation

External operational context was simulated to enrich machine telemetry.

Added contextual signals:

- Ambient Humidity
- Energy Load Index
- Production Demand
- Days Since Maintenance
- Shift Indicator

These features provide additional information beyond sensor telemetry alone.

## Context Interaction Features

To model the interaction between machine telemetry and external operating conditions, interaction features are created.

These features capture relationships that may not be visible when telemetry and context variables are analyzed independently.

In [52]:
df = pd.read_csv("../data/processed/context_base.csv")

print(df.shape)

(9988, 49)


In [53]:
df["torque_x_load"] = df["Torque [Nm]"] * df["energy_load_index"]

In [54]:
df["wear_x_demand"] = df["Tool wear [min]"] * df["production_demand"]

In [55]:
df["temp_x_humidity"] = df["Process temperature [K]"] * df["ambient_humidity"]

In [56]:
print(df.shape)

(9988, 52)


In [57]:
interaction_cols = ["torque_x_load", "wear_x_demand", "temp_x_humidity"]

df[interaction_cols].describe()

,torque_x_load,wear_x_demand,temp_x_humidity
count,9988.000000,9988.000000,9988.000000
mean,2816.544003,10798.564948,20143.508064
std,1003.715958,7336.393252,3110.211184
min,313.175711,0.000000,7987.981620
25%,2059.634317,4792.912990,18060.889531
50%,2706.813076,9736.495514,20147.675080
75%,3487.991127,15580.382526,22208.560865
max,7288.871811,37623.236801,32258.779464


In [58]:
corr_df = df[interaction_cols + ["Machine failure"]].corr()

corr_df["Machine failure"]

torque_x_load      0.137775
wear_x_demand      0.097877
temp_x_humidity   -0.015107
Machine failure    1.000000
Name: Machine failure, dtype: float64

In [59]:
df.to_csv("../data/processed/context_fused_dataset.csv", index=False)

print("Context-fused dataset saved.")

Context-fused dataset saved.


### Interaction Feature Findings

Three contextual interaction features were created:

- Torque × Energy Load
- Tool Wear × Production Demand
- Process Temperature × Ambient Humidity

These variables capture relationships between machine condition and operating environment, forming the basis of contextual predictive maintenance modeling.

In [60]:
df = pd.read_csv("../data/processed/context_fused_dataset.csv")

print(df.shape)

(9988, 52)


In [61]:
missing_values = df.isnull().sum().sum()

print("Total Missing Values:", missing_values)

Total Missing Values: 0


In [62]:
context_cols = [
    "ambient_humidity",
    "energy_load_index",
    "production_demand",
    "days_since_maintenance",
    "shift",
]

interaction_cols = ["torque_x_load", "wear_x_demand", "temp_x_humidity"]

print("Context Features:", len(context_cols))

print("Interaction Features:", len(interaction_cols))

Context Features: 5
Interaction Features: 3


In [63]:
validation_corr = df[context_cols + interaction_cols + ["Machine failure"]].corr()

validation_corr["Machine failure"].sort_values(ascending=False)

Machine failure           1.000000
torque_x_load             0.137775
wear_x_demand             0.097877
production_demand         0.020676
energy_load_index         0.005383
shift                     0.001904
days_since_maintenance   -0.001880
temp_x_humidity          -0.015107
ambient_humidity         -0.016185
Name: Machine failure, dtype: float64

In [64]:
context_summary = pd.DataFrame(
    {
        "Feature Type": ["Telemetry", "Context", "Interaction"],
        "Count": [44, len(context_cols), len(interaction_cols)],
    }
)

context_summary

,Feature Type,Count
0,Telemetry,44
1,Context,5
2,Interaction,3


In [65]:
print(f"Rows: {df.shape[0]}")

print(f"Columns: {df.shape[1]}")

Rows: 9988
Columns: 52


In [66]:
df.to_csv("../data/processed/context_fused_validated.csv", index=False)

print("Validated context-fused dataset saved.")

Validated context-fused dataset saved.
